# Channel Video Geometry (Shared 20D Space)

This notebook builds geometric artifacts for each channel's video cloud in one shared 20-dimensional embedding space.

> **Interpretation note:** This notebook measures the geometry of channel video clouds before any engagement modeling. The purpose is to test whether channels behave like compact isotropic clouds, elongated anisotropic clouds, or multimodal semantic manifolds. Later notebooks can overlay engagement and the subscription graph onto these shared-space geometric artifacts.


## 1) Setup: imports, paths, reproducibility, and artifact folders

In [ ]:

import ast
import json
import math
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from sklearn.neighbors import NearestNeighbors

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Conservative defaults; tune here, not deep in the notebook.
CONFIG = {
    "embedding_dim": 20,
    "input_candidates": [
        Path("/content/drive/MyDrive/Graphiko/exports/video_embeddings.parquet"),
        Path("/content/drive/MyDrive/Graphiko/exports/video_embeddings.csv"),
        Path("data/video_embeddings.parquet"),
        Path("data/video_embeddings.csv"),
    ],
    "output_root": Path("artifacts/channel_video_geometry"),
    "time_decay_half_life_days": 180.0,
    "knn_k": 5,
    "min_prior_for_novelty": 5,
    "hdbscan_min_cluster_size": 6,
    "hdbscan_min_samples": 4,
    "mode_min_size_for_centroid_distance": 3,
    "rolling_drift_min_history": 3,
}

OUTPUT = CONFIG["output_root"]
PLOTS = OUTPUT / "plots"
PLOTS_CHANNEL = PLOTS / "per_channel"
PLOTS_CROSS = PLOTS / "cross_channel"

for d in [OUTPUT, PLOTS, PLOTS_CHANNEL, PLOTS_CROSS]:
    d.mkdir(parents=True, exist_ok=True)

pd.options.display.max_columns = 200
sns.set_theme(style="whitegrid")
print("Seed:", RANDOM_SEED)
print("Output root:", OUTPUT.resolve())


## 2) Load exported embeddings and metadata

In [ ]:

def pick_input_path(candidates):
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "No input file found. Checked: " + ", ".join(str(p) for p in candidates)
    )


def parse_embedding_value(x):
    if isinstance(x, np.ndarray):
        return x.astype(float)
    if isinstance(x, list):
        return np.asarray(x, dtype=float)
    if isinstance(x, tuple):
        return np.asarray(list(x), dtype=float)
    if isinstance(x, str):
        s = x.strip()
        try:
            # Handles JSON list strings and Python literal list strings.
            v = ast.literal_eval(s)
        except Exception as e:
            raise ValueError(f"Could not parse embedding string: {s[:80]}...") from e
        return np.asarray(v, dtype=float)
    raise TypeError(f"Unsupported embedding type: {type(x)}")


def find_embedding_column(df):
    candidates = ["embedding", "vector", "videoEmbedding", "video_embedding", "emb"]
    for c in candidates:
        if c in df.columns:
            return c
    for c in df.columns:
        sample = df[c].dropna().head(5)
        if len(sample) == 0:
            continue
        ok = 0
        for v in sample:
            if isinstance(v, (list, tuple, np.ndarray)):
                ok += 1
            elif isinstance(v, str) and (v.strip().startswith("[") and v.strip().endswith("]")):
                ok += 1
        if ok >= max(1, len(sample) - 1):
            return c
    raise KeyError("No embedding column found. Expected one of: embedding/vector/videoEmbedding/video_embedding/emb")


input_path = pick_input_path(CONFIG["input_candidates"])
print("Loading:", input_path)

if input_path.suffix.lower() == ".parquet":
    raw = pd.read_parquet(input_path)
elif input_path.suffix.lower() == ".csv":
    raw = pd.read_csv(input_path)
else:
    raise ValueError(f"Unsupported file extension: {input_path.suffix}")

required_cols = {"videoId", "channelId", "title", "publishedAt"}
missing_required = sorted(required_cols - set(raw.columns))
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

embedding_col = find_embedding_column(raw)
print("Embedding column:", embedding_col)

if "channelTitle" not in raw.columns:
    raw["channelTitle"] = np.nan
if "channelDescription" not in raw.columns:
    raw["channelDescription"] = np.nan

base_cols = ["videoId", "channelId", "title", "publishedAt", "channelTitle", "channelDescription", embedding_col]
df = raw[base_cols].copy()
df = df.rename(columns={embedding_col: "embedding_raw"})


## 3) Validate data and build one global dataframe

In [ ]:

df["embedding"] = df["embedding_raw"].apply(parse_embedding_value)

bad_len = df["embedding"].apply(lambda v: len(v) != CONFIG["embedding_dim"])
if bad_len.any():
    bad_ids = df.loc[bad_len, "videoId"].head(10).tolist()
    raise ValueError(f"Found embeddings with wrong dimension (expected {CONFIG['embedding_dim']}), example videoIds: {bad_ids}")

bad_numeric = df["embedding"].apply(lambda v: not np.issubdtype(v.dtype, np.number))
if bad_numeric.any():
    raise ValueError("Found non-numeric embeddings")

bad_finite = df["embedding"].apply(lambda v: not np.isfinite(v).all())
if bad_finite.any():
    bad_ids = df.loc[bad_finite, "videoId"].head(10).tolist()
    raise ValueError(f"Found non-finite embeddings, example videoIds: {bad_ids}")

if df["channelId"].isna().any():
    raise ValueError("Missing channelId for some videos")

df["publishedAt"] = pd.to_datetime(df["publishedAt"], utc=True, errors="coerce")
if df["publishedAt"].isna().any():
    bad_ids = df.loc[df["publishedAt"].isna(), "videoId"].head(10).tolist()
    raise ValueError(f"Invalid publishedAt values, example videoIds: {bad_ids}")

video_df = df[["videoId", "channelId", "title", "publishedAt", "channelTitle", "channelDescription", "embedding"]].copy()
video_df["embedding"] = video_df["embedding"].apply(lambda x: np.asarray(x, dtype=float))

print("Videos:", len(video_df))
print("Channels:", video_df["channelId"].nunique())
video_df.head(3)


## 4) Global shared-space coordinates for visualization only (UMAP + optional PCA)

In [ ]:

X_all = np.vstack(video_df["embedding"].to_numpy())

global_pca = PCA(n_components=2, random_state=RANDOM_SEED)
X_pca2 = global_pca.fit_transform(X_all)
video_df["pca2_x"] = X_pca2[:, 0]
video_df["pca2_y"] = X_pca2[:, 1]

try:
    import umap
except ImportError as e:
    raise ImportError("umap-learn is required for visualization. Install with `pip install umap-learn`.") from e

global_umap = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=RANDOM_SEED,
)
X_umap2 = global_umap.fit_transform(X_all)
video_df["umap2_x"] = X_umap2[:, 0]
video_df["umap2_y"] = X_umap2[:, 1]

print("Global UMAP fit complete.")


## 5) Geometry helpers (shared 20D calculations)

In [ ]:

def safe_cosine_distances(X, Y=None):
    return pairwise_distances(X, X if Y is None else Y, metric="cosine")


def compute_medoid(X):
    D = pairwise_distances(X, metric="euclidean")
    idx = np.argmin(D.sum(axis=1))
    return idx, X[idx]


def recency_weighted_centroid(X, published_at, half_life_days=180.0):
    tmax = published_at.max()
    age_days = (tmax - published_at).dt.total_seconds().to_numpy() / 86400.0
    w = 0.5 ** (age_days / half_life_days)
    w = np.clip(w, 1e-9, None)
    w = w / w.sum()
    c = (X * w[:, None]).sum(axis=0)
    return c, w


def effective_rank(evals, eps=1e-12):
    s = np.asarray(evals, dtype=float)
    s = np.clip(s, eps, None)
    p = s / s.sum()
    return float(np.exp(-(p * np.log(p)).sum()))


def fit_shrinkage_covariance(X):
    model = LedoitWolf().fit(X)
    cov = model.covariance_
    precision = model.precision_
    evals = np.linalg.eigvalsh(cov)
    evals = np.sort(np.clip(evals, 0.0, None))[::-1]
    return model, cov, precision, evals


def mahalanobis_to_center(X, center, precision):
    diff = X - center
    vals = np.einsum("ij,jk,ik->i", diff, precision, diff)
    vals = np.clip(vals, 0.0, None)
    return np.sqrt(vals)


def knn_avg_distance(X, k=5):
    n = X.shape[0]
    if n <= 1:
        return np.full(n, np.nan)
    k_eff = min(max(1, k), n - 1)
    nn = NearestNeighbors(n_neighbors=k_eff + 1, metric="euclidean")
    nn.fit(X)
    dists, _ = nn.kneighbors(X)
    return dists[:, 1:].mean(axis=1)


def percentile_rank_desc(values):
    s = pd.Series(values)
    return (1.0 - s.rank(method="average", pct=True)).to_numpy()


def centroid_shift_first_vs_second_half(X):
    n = len(X)
    if n < 2:
        return np.nan
    mid = n // 2
    a = X[:mid].mean(axis=0)
    b = X[mid:].mean(axis=0)
    return float(np.linalg.norm(a - b))


## 6) Per-channel and per-video geometry

In [ ]:

try:
    import hdbscan
except ImportError as e:
    raise ImportError("hdbscan is required for per-channel mode detection. Install with `pip install hdbscan`.") from e

channel_rows = []
mode_rows = []
video_rows = []
eigenspectra = {}

for channel_id, g in video_df.groupby("channelId", sort=False):
    g = g.sort_values("publishedAt").copy()
    X = np.vstack(g["embedding"].to_numpy())
    n = X.shape[0]

    centroid = X.mean(axis=0)
    medoid_idx, medoid = compute_medoid(X)
    rw_centroid, _ = recency_weighted_centroid(X, g["publishedAt"], CONFIG["time_decay_half_life_days"])

    lw_model, cov, precision, evals = fit_shrinkage_covariance(X)
    var_total = float(evals.sum())
    top1_share = float(evals[0] / var_total) if var_total > 0 else np.nan
    top3_share = float(evals[:3].sum() / var_total) if var_total > 0 else np.nan
    eff_rank = effective_rank(evals) if var_total > 0 else np.nan

    pair_cos = pairwise_distances(X, metric="cosine")
    iu = np.triu_indices(n, k=1)
    pair_cos_vals = pair_cos[iu] if n > 1 else np.array([np.nan])

    dist_euc_centroid = np.linalg.norm(X - centroid, axis=1)
    mean_pair_cos = float(np.nanmean(pair_cos_vals))
    med_pair_cos = float(np.nanmedian(pair_cos_vals))
    mean_euc_centroid = float(np.nanmean(dist_euc_centroid))
    med_euc_centroid = float(np.nanmedian(dist_euc_centroid))

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=CONFIG["hdbscan_min_cluster_size"],
        min_samples=CONFIG["hdbscan_min_samples"],
        metric="euclidean",
        prediction_data=False,
    )
    labels = clusterer.fit_predict(X)
    non_noise = labels[labels >= 0]
    if len(non_noise) == 0:
        labels = np.zeros(n, dtype=int)
        noise_share = 0.0
    else:
        noise_share = float((labels < 0).mean())

    unique_modes = sorted([m for m in np.unique(labels) if m >= 0])
    if len(unique_modes) == 0:
        unique_modes = [0]
        labels = np.zeros(n, dtype=int)

    mode_counts = pd.Series(labels).value_counts(dropna=False).sort_index().to_dict()
    mode_counts_clean = {int(k): int(v) for k, v in mode_counts.items() if int(k) >= 0}
    largest_mode_share = float(max(mode_counts_clean.values()) / n) if mode_counts_clean else 1.0

    mode_centroids = {}
    mode_medoids = {}
    for m in unique_modes:
        idx = np.where(labels == m)[0]
        Xm = X[idx]
        mode_centroids[int(m)] = Xm.mean(axis=0)
        med_i, med_v = compute_medoid(Xm)
        mode_medoids[int(m)] = med_v
        mode_rows.append({
            "channelId": channel_id,
            "mode_label": int(m),
            "mode_count": int(len(idx)),
            "mode_share": float(len(idx) / n),
            "mode_centroid": mode_centroids[int(m)].tolist(),
            "mode_medoid": mode_medoids[int(m)].tolist(),
        })

    cos_to_centroid = safe_cosine_distances(X, centroid.reshape(1, -1)).ravel()
    cos_to_medoid = safe_cosine_distances(X, medoid.reshape(1, -1)).ravel()
    euc_to_centroid = np.linalg.norm(X - centroid, axis=1)
    mahal_to_centroid = mahalanobis_to_center(X, centroid, precision)
    euc_to_rw_centroid = np.linalg.norm(X - rw_centroid, axis=1)

    knn_avg = knn_avg_distance(X, k=CONFIG["knn_k"])
    density_pct = percentile_rank_desc(knn_avg)

    temporal_novelty = np.full(n, np.nan)
    dist_to_prior_centroid = np.full(n, np.nan)
    rolling_drift = np.full(n, np.nan)

    running_centroid = np.zeros(X.shape[1], dtype=float)
    for i in range(n):
        if i > 0:
            prior_centroid = X[:i].mean(axis=0)
            dist_to_prior_centroid[i] = np.linalg.norm(X[i] - prior_centroid)
            if i >= CONFIG["min_prior_for_novelty"]:
                temporal_novelty[i] = dist_to_prior_centroid[i]

        c_t = X[: i + 1].mean(axis=0)
        if i > 0:
            rolling_drift[i] = np.linalg.norm(c_t - running_centroid)
        running_centroid = c_t

    dist_to_mode_centroid = np.full(n, np.nan)
    for m in unique_modes:
        idx = np.where(labels == m)[0]
        if len(idx) >= CONFIG["mode_min_size_for_centroid_distance"]:
            c_m = mode_centroids[int(m)]
            dist_to_mode_centroid[idx] = np.linalg.norm(X[idx] - c_m, axis=1)

    long_run_dist = np.linalg.norm(X - centroid, axis=1)
    half_shift = centroid_shift_first_vs_second_half(X)

    for i, (_, r) in enumerate(g.iterrows()):
        video_rows.append({
            "videoId": r["videoId"],
            "channelId": channel_id,
            "title": r["title"],
            "publishedAt": r["publishedAt"],
            "cosine_distance_to_centroid": float(cos_to_centroid[i]),
            "euclidean_distance_to_centroid": float(euc_to_centroid[i]),
            "cosine_distance_to_medoid": float(cos_to_medoid[i]),
            "shrinkage_mahalanobis_distance": float(mahal_to_centroid[i]),
            "knn_avg_distance_within_channel": float(knn_avg[i]) if np.isfinite(knn_avg[i]) else np.nan,
            "local_density_percentile": float(density_pct[i]) if np.isfinite(density_pct[i]) else np.nan,
            "temporal_novelty_score": float(temporal_novelty[i]) if np.isfinite(temporal_novelty[i]) else np.nan,
            "distance_to_recency_weighted_centroid": float(euc_to_rw_centroid[i]),
            "mode_label": int(labels[i]) if int(labels[i]) >= 0 else 0,
            "distance_to_mode_centroid": float(dist_to_mode_centroid[i]) if np.isfinite(dist_to_mode_centroid[i]) else np.nan,
            "distance_to_long_run_centroid": float(long_run_dist[i]),
            "distance_to_prior_history_centroid": float(dist_to_prior_centroid[i]) if np.isfinite(dist_to_prior_centroid[i]) else np.nan,
            "rolling_centroid_drift": float(rolling_drift[i]) if np.isfinite(rolling_drift[i]) else np.nan,
        })

    channel_rows.append({
        "channelId": channel_id,
        "channelTitle": g["channelTitle"].dropna().iloc[0] if g["channelTitle"].notna().any() else None,
        "channelDescription": g["channelDescription"].dropna().iloc[0] if g["channelDescription"].notna().any() else None,
        "n_videos": int(n),
        "centroid": centroid.tolist(),
        "medoid": medoid.tolist(),
        "recency_weighted_centroid": rw_centroid.tolist(),
        "top1_variance_share": top1_share,
        "top3_variance_share": top3_share,
        "effective_rank": eff_rank,
        "mean_pairwise_cosine_distance": mean_pair_cos,
        "median_pairwise_cosine_distance": med_pair_cos,
        "mean_euclidean_distance_to_centroid": mean_euc_centroid,
        "median_euclidean_distance_to_centroid": med_euc_centroid,
        "n_modes": int(len(unique_modes)),
        "largest_mode_share": largest_mode_share,
        "noise_share": noise_share,
        "per_mode_counts": json.dumps(mode_counts_clean),
        "centroid_shift_first_vs_second_half": float(half_shift) if np.isfinite(half_shift) else np.nan,
        "mean_temporal_novelty": float(np.nanmean(temporal_novelty)) if np.isfinite(np.nanmean(temporal_novelty)) else np.nan,
        "median_temporal_novelty": float(np.nanmedian(temporal_novelty)) if np.isfinite(np.nanmedian(temporal_novelty)) else np.nan,
        "mean_rolling_centroid_drift": float(np.nanmean(rolling_drift)) if np.isfinite(np.nanmean(rolling_drift)) else np.nan,
    })

    eigenspectra[str(channel_id)] = {
        "channelId": channel_id,
        "n_videos": int(n),
        "eigenvalues": [float(v) for v in evals],
        "top1_variance_share": top1_share,
        "top3_variance_share": top3_share,
        "effective_rank": eff_rank,
    }

channel_summary = pd.DataFrame(channel_rows)
mode_summary = pd.DataFrame(mode_rows)
video_scores = pd.DataFrame(video_rows)

coords = video_df[["videoId", "umap2_x", "umap2_y", "pca2_x", "pca2_y"]].copy()
video_scores = video_scores.merge(coords, on="videoId", how="left")

print("channel_summary shape:", channel_summary.shape)
print("mode_summary shape:", mode_summary.shape)
print("video_scores shape:", video_scores.shape)


## 7) Cross-channel summary tables and ranked exports

In [ ]:

channel_summary["compactness_score"] = channel_summary["mean_euclidean_distance_to_centroid"]
channel_summary["anisotropy_score"] = channel_summary["top1_variance_share"]
channel_summary["multimodality_score"] = 1.0 - channel_summary["largest_mode_share"]
channel_summary["exploratory_novelty_score"] = channel_summary["mean_temporal_novelty"]
channel_summary["temporal_drift_score"] = channel_summary["centroid_shift_first_vs_second_half"]

video_scores.to_csv(OUTPUT / "video_geometry_scores.csv", index=False)
channel_summary.to_csv(OUTPUT / "channel_geometry_summary.csv", index=False)
mode_summary.to_csv(OUTPUT / "channel_mode_summary.csv", index=False)

with open(OUTPUT / "channel_covariance_eigenspectra.json", "w", encoding="utf-8") as f:
    json.dump(eigenspectra, f, indent=2)

(channel_summary.sort_values("anisotropy_score", ascending=False)
 .to_csv(OUTPUT / "channel_summary_most_anisotropic.csv", index=False))
(channel_summary.sort_values("multimodality_score", ascending=False)
 .to_csv(OUTPUT / "channel_summary_most_multimodal.csv", index=False))
(channel_summary.sort_values("compactness_score", ascending=True)
 .to_csv(OUTPUT / "channel_summary_most_compact.csv", index=False))
(channel_summary.sort_values("exploratory_novelty_score", ascending=False)
 .to_csv(OUTPUT / "channel_summary_most_exploratory_by_novelty.csv", index=False))
(channel_summary.sort_values("temporal_drift_score", ascending=False)
 .to_csv(OUTPUT / "channel_summary_most_drifted_over_time.csv", index=False))

print("Saved core CSV/JSON artifacts to", OUTPUT)
channel_summary.head(10)


## 8) Per-channel visual outputs

In [ ]:

for channel_id, g in video_scores.groupby("channelId", sort=False):
    out_dir = PLOTS_CHANNEL / str(channel_id)
    out_dir.mkdir(parents=True, exist_ok=True)

    gg = g.sort_values("publishedAt").copy()
    gg["publishedAt"] = pd.to_datetime(gg["publishedAt"], utc=True)

    fig, ax = plt.subplots(figsize=(6, 5))
    cvals = (gg["publishedAt"].astype("int64") // 10**9).to_numpy()
    sc = ax.scatter(gg["umap2_x"], gg["umap2_y"], c=cvals, cmap="viridis", s=28)
    ax.set_title(f"{channel_id} - UMAP colored by time")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    plt.colorbar(sc, ax=ax, label="publishedAt (unix)")
    fig.tight_layout()
    fig.savefig(out_dir / "umap_time.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.scatterplot(data=gg, x="umap2_x", y="umap2_y", hue="mode_label", palette="tab10", ax=ax, s=35)
    ax.set_title(f"{channel_id} - UMAP colored by mode")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    fig.tight_layout()
    fig.savefig(out_dir / "umap_mode.png", dpi=150)
    plt.close(fig)

    evals = eigenspectra[str(channel_id)]["eigenvalues"]
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(range(1, len(evals) + 1), evals, marker="o")
    ax.set_title(f"{channel_id} - Covariance eigenspectrum")
    ax.set_xlabel("Eigenvalue index")
    ax.set_ylabel("Eigenvalue")
    fig.tight_layout()
    fig.savefig(out_dir / "eigenspectrum_scree.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.histplot(gg["temporal_novelty_score"].dropna(), bins=20, ax=ax)
    ax.set_title(f"{channel_id} - Temporal novelty histogram")
    ax.set_xlabel("Temporal novelty score")
    fig.tight_layout()
    fig.savefig(out_dir / "novelty_histogram.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(gg["publishedAt"], gg["temporal_novelty_score"], marker="o", linewidth=1)
    ax.set_title(f"{channel_id} - Temporal novelty timeline")
    ax.set_xlabel("publishedAt")
    ax.set_ylabel("Temporal novelty score")
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(out_dir / "novelty_timeline.png", dpi=150)
    plt.close(fig)

print("Per-channel plots saved under", PLOTS_CHANNEL)


## 9) Cross-channel visual outputs

In [ ]:

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(channel_summary["effective_rank"].dropna(), bins=20, ax=ax)
ax.set_title("Effective rank across channels")
ax.set_xlabel("effective_rank")
fig.tight_layout()
fig.savefig(PLOTS_CROSS / "hist_effective_rank.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(channel_summary["largest_mode_share"].dropna(), bins=20, ax=ax)
ax.set_title("Largest mode share across channels")
ax.set_xlabel("largest_mode_share")
fig.tight_layout()
fig.savefig(PLOTS_CROSS / "hist_largest_mode_share.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=channel_summary,
    x="compactness_score",
    y="anisotropy_score",
    size="n_videos",
    legend=False,
    ax=ax,
)
ax.set_title("Compactness vs anisotropy")
ax.set_xlabel("compactness (mean Euclidean distance to centroid)")
ax.set_ylabel("anisotropy (top-1 variance share)")
fig.tight_layout()
fig.savefig(PLOTS_CROSS / "scatter_compactness_vs_anisotropy.png", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(
    data=channel_summary,
    x="temporal_drift_score",
    y="multimodality_score",
    size="n_videos",
    legend=False,
    ax=ax,
)
ax.set_title("Temporal drift vs multimodality")
ax.set_xlabel("temporal drift (first-half vs second-half centroid shift)")
ax.set_ylabel("multimodality (1 - largest mode share)")
fig.tight_layout()
fig.savefig(PLOTS_CROSS / "scatter_temporal_drift_vs_multimodality.png", dpi=150)
plt.close(fig)

metric_cols = [
    "compactness_score",
    "anisotropy_score",
    "effective_rank",
    "largest_mode_share",
    "multimodality_score",
    "exploratory_novelty_score",
    "temporal_drift_score",
]
heat = channel_summary[["channelId"] + metric_cols].dropna(subset=metric_cols, how="all").copy()
for c in metric_cols:
    std = heat[c].std(ddof=0)
    if pd.isna(std) or std == 0:
        heat[c + "_z"] = 0.0
    else:
        heat[c + "_z"] = (heat[c] - heat[c].mean()) / std

z_cols = [c + "_z" for c in metric_cols]
heat_mat = heat.set_index("channelId")[z_cols]

fig, ax = plt.subplots(figsize=(10, max(4, 0.25 * len(heat_mat))))
sns.heatmap(heat_mat, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Channel-by-metric z-scores")
fig.tight_layout()
fig.savefig(PLOTS_CROSS / "heatmap_channel_metric_zscores.png", dpi=150)
plt.close(fig)

print("Cross-channel plots saved under", PLOTS_CROSS)


## 10) Save notebook summary metadata

In [ ]:

summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "input_path": str(input_path),
    "output_root": str(OUTPUT),
    "n_videos": int(len(video_scores)),
    "n_channels": int(channel_summary["channelId"].nunique()),
    "config": {k: str(v) if isinstance(v, Path) else v for k, v in CONFIG.items()},
    "artifacts": {
        "video_geometry_scores_csv": str(OUTPUT / "video_geometry_scores.csv"),
        "channel_geometry_summary_csv": str(OUTPUT / "channel_geometry_summary.csv"),
        "channel_mode_summary_csv": str(OUTPUT / "channel_mode_summary.csv"),
        "channel_covariance_eigenspectra_json": str(OUTPUT / "channel_covariance_eigenspectra.json"),
        "plots_per_channel_dir": str(PLOTS_CHANNEL),
        "plots_cross_channel_dir": str(PLOTS_CROSS),
    },
}

with open(OUTPUT / "notebook_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

summary
